# A3.7 · Runtime containment levers

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

Builds on **[A3.6 · Egress control for agents](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**.

| | |
|---|---|
| Open-source tooling | Falco, Kyverno, kagent |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Every control before this one is preventive. This lesson is about what you have
left when prevention has failed and an agent is actively doing damage.

There are four levers, and they differ on two axes that matter during an
incident: **how fast they take effect**, and **what they leave running**.

| Lever | Speed | What it misses |
|---|---|---|
| Kill the process | seconds | It restarts. The credential still works. |
| Network quarantine | seconds | Stops egress, not local damage |
| **Revoke the identity** | seconds | Nothing — the agent cannot act anywhere, even after restart |
| Rotate the credential | minutes | Correct, but slow and breaks bystanders |

Revoking the identity is almost always the right first lever, and it is only
available if A2 was done: an agent with its own separately-revocable identity
can be stopped without stopping anything else. That is the operational payoff of
the entire identity track.

The second thing this lesson establishes is a number: **how much damage happens
while containment waits for a human approval**. It is usually the argument that
gets automated containment funded.

## 2 · Demo — the four levers, timed

In [ ]:
from dataclasses import dataclass, field
import time

@dataclass
class Fleet:
    """Three agents sharing infrastructure, each with its own identity."""
    running: set = field(default_factory=lambda: {"triage-agent", "patch-agent",
                                                  "deploy-agent"})
    identities_valid: set = field(default_factory=lambda: {"triage-agent",
                                                           "patch-agent", "deploy-agent"})
    shared_credential_valid: bool = True

    def can_act(self, agent):
        return (agent in self.running and agent in self.identities_valid
                and self.shared_credential_valid)

    # --- the levers -------------------------------------------------------
    def kill_process(self, agent, restarts=True):
        self.running.discard(agent)
        detail = "process killed"
        if restarts:                      # supervisors restart things. That is their job.
            self.running.add(agent)
            detail += "; supervisor restarted it — identity still valid"
        return 2, detail
    def quarantine(self, agent):
        return 5, "egress blocked; local filesystem actions continue"
    def revoke_identity(self, agent):
        self.identities_valid.discard(agent)
        return 12, "identity revoked; agent cannot act even after restart"
    def rotate_shared_credential(self):
        self.shared_credential_valid = False
        return 420, "credential rotated; EVERY consumer is now broken"

for lever in ("kill_process", "quarantine", "revoke_identity", "rotate_shared_credential"):
    f = Fleet()
    secs, detail = (getattr(f, lever)("patch-agent") if lever != "rotate_shared_credential"
                    else f.rotate_shared_credential())
    still = sorted(a for a in ("triage-agent", "patch-agent", "deploy-agent") if f.can_act(a))
    print(f"{lever:26s} {secs:>4}s  target still acts: "
          f"{f.can_act('patch-agent')!s:5s}  others up: {[a for a in still if a!='patch-agent']}")
    print(f"{'':26s}       {detail}")

## 3 · Where it breaks — the human in the containment path

The lever is fast. The decision to pull it is not, if it needs an approval. Here is the cost, in the only unit that matters: **actions the agent completed while you were waiting.**

In [ ]:
def containment_race(actions_per_min, human_minutes, auto_seconds=12):
    manual = actions_per_min * human_minutes
    auto   = actions_per_min * (auto_seconds / 60)
    return {"manual": round(manual), "auto": round(auto),
            "ratio": round(manual / max(auto, 1e-9), 1)}

print(f"{'agent rate':>14}{'human 8min':>13}{'automated 12s':>16}{'ratio':>8}")
print("-" * 52)
for rate in (60, 300, 1200):
    r = containment_race(rate, human_minutes=8)
    print(f"{rate:>10}/min{r['manual']:>13}{r['auto']:>16}{r['ratio']:>8}×")

print("\nAt 300 actions/min an 8-minute approval costs 2,400 further actions")
print("against 60 for automated revocation. That ratio is the funding argument.")

## 4 · The control — pre-authorised revocation for non-human identities

The asymmetry to exploit: revoking a *human's* access needs care, because a false positive locks out a person mid-shift. Revoking a *non-human* identity is cheap to get wrong — the agent re-requests, or an on-call re-enables it in a minute.

So the policy can be different, and should be: **automated revocation for NHIs on high-confidence signals, no human in the path.**

In [ ]:
RULES = {
 "reached the cloud metadata service": {"confidence": 0.99, "auto": True},
 "read a path matching */.ssh/*":       {"confidence": 0.97, "auto": True},
 "egress to an unlisted host":          {"confidence": 0.90, "auto": True},
 "tool-call rate 20× baseline":         {"confidence": 0.75, "auto": True},
 "unusual working hours":               {"confidence": 0.30, "auto": False},
}
THRESHOLD = 0.70

def respond(signal, subject_is_human):
    rule = RULES[signal]
    if subject_is_human:
        return "page on-call — human lockout needs a person to confirm"
    if rule["confidence"] >= THRESHOLD and rule["auto"]:
        return f"AUTO-REVOKE (confidence {rule['confidence']:.2f})"
    return f"alert only (confidence {rule['confidence']:.2f} < {THRESHOLD})"

for sig in RULES:
    print(f"{sig:38s} NHI: {respond(sig, False):44s}")
print()
print(f"{'same signal, human subject':38s}     {respond('read a path matching */.ssh/*', True)}")

In [ ]:
# Verify: end-to-end. Signal fires, identity revoked, agent cannot act,
# and the rest of the fleet is unaffected.
f = Fleet()
print("before:", {a: f.can_act(a) for a in sorted(f.running)})

signal = "reached the cloud metadata service"
action = respond(signal, subject_is_human=False)
print(f"\nsignal: {signal}\naction: {action}")
assert action.startswith("AUTO-REVOKE")
secs, detail = f.revoke_identity("patch-agent")

print(f"after {secs}s:", {a: f.can_act(a) for a in sorted(f.running | {'patch-agent'})})
assert not f.can_act("patch-agent")
assert f.can_act("triage-agent") and f.can_act("deploy-agent")
print("\nContained agent stopped. Two other agents unaffected. No human waited.")
print("This only works because A2 gave each agent its own revocable identity.")

## What you just proved

Killing the process leaves the agent acting after a supervisor restart; quarantine leaves local actions running; identity revocation stops it in 12s with the other agents up; credential rotation takes 420s and breaks everything. The race table shows 2,400 vs 60 actions at 300/min. Four of five signals auto-revoke for NHIs and none do for a human subject. The end-to-end check confirms only the target stops.

## Your turn

Time your own revocation path: from the decision to the agent's next call actually failing. Most teams find it is minutes, and that the slow part is finding the right console — not the revocation itself.

---

**Next → [A3.8 · Environment separation](https://spbreed.github.io/cyber-commons/lessons/A3.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*